# Homework: Extending the RNN Language Detector

This homework builds directly on the RNN language detector from the lecture. You will **not** build anything from scratch — instead you will make targeted adjustments to the existing code.

**Three tasks:**
1. Improve the training loop with a validation set, early stopping, and learning rate scheduling.
2. Turn the sequence classifier into a token-level classifier (a first step towards NER).
3. Stack a second and third RNN layer on top of the first.

Estimated time: **30–60 minutes**

---

Cells that require your input use a `TODO()` helper that raises an error until you fill them in:

In [ ]:
def TODO(msg):
    raise NotImplementedError(f"TODO not completed: {msg}")

---

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import matplotlib.pyplot as plt
from IPython import display
from nltk import word_tokenize 

In [ ]:
# Data loading (identical to the lecture notebook)
train_df = pd.read_csv("../data/lang_train.csv")
test_df  = pd.read_csv("../data/lang_test.csv")

LANGUAGES = ["de", "es", "en"]
train_df = train_df[train_df["labels"].isin(LANGUAGES)]
test_df  = test_df[test_df["labels"].isin(LANGUAGES)]

texts_train  = list(train_df["text"].values)
texts_test   = list(test_df["text"].values)
labels_train = list(train_df["labels"].values)
labels_test  = list(test_df["labels"].values)

unique_labels = sorted(list(set(labels_train + labels_test)))
label_to_idx  = {label: i for i, label in enumerate(unique_labels)}
idx_to_label  = {i: label for label, i in label_to_idx.items()}
NUM_CLASSES   = len(unique_labels)

In [ ]:
# Vocabulary and encoder (identical to the lecture notebook)
word_corpus = []
for text in texts_train:
    word_corpus.extend([w for w in word_tokenize(text.lower()) if w.isalpha()])
    
word_vocab = sorted(list(set(word_corpus)))
word_to_id = {gram: i+1 for i, gram in enumerate(word_vocab)}
VOCAB_SIZE  = len(word_vocab) + 1
MAX_SEQ_LEN = 10

def word_encoder(text, max_seq_len=15):
    id_seq = [0] * max_seq_len
    for i, word in enumerate(word_tokenize(text)):
        if i >= max_seq_len:
            break
        if word.isalpha():
            id_seq[i] = word_to_id.get(word.lower(), 0)
    return id_seq

In [ ]:
# Dataset (identical to the lecture notebook)
class SentenceDataset(Dataset):
    def __init__(self, texts, labels, label_mapping, max_seq_len=MAX_SEQ_LEN):
        self.texts        = texts
        self.labels       = labels
        self.label_to_idx = label_mapping
        self.max_seq_len  = max_seq_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text     = self.texts[idx]
        label_id = self.label_to_idx[self.labels[idx]]
        token_ids = torch.tensor(word_encoder(text, self.max_seq_len)).long()
        return text, token_ids, label_id

full_train_dataset = SentenceDataset(texts_train, labels_train, label_to_idx)
test_dataset       = SentenceDataset(texts_test,  labels_test,  label_to_idx)

In [ ]:
# Model (identical to the lecture notebook)
EMBED_DIM  = 32
HIDDEN_DIM = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.i2h        = nn.Linear(embed_dim,  hidden_dim, bias=True)
        self.h2h        = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len = x.shape
        embedded = self.dropout(self.embedding(x))
        h_t = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        all_hidden_states = []
        for t in range(seq_len):
            x_t    = embedded[:, t, :]
            mask   = (x[:, t] != 0).float().unsqueeze(1)
            h_next = torch.tanh(self.i2h(x_t) + self.h2h(h_t))
            h_t    = mask * h_next + (1 - mask) * h_t
            all_hidden_states.append(h_t.unsqueeze(1))
        output = torch.cat(all_hidden_states, dim=1)  # (batch, seq_len, hidden_dim)
        return output, h_t

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.rnn        = RNNEncoder(vocab_size, embed_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, token_ids):
        _, h_T = self.rnn(token_ids)
        return self.classifier(h_T)  # (batch, num_classes)

---

## Task 1 · Better Training: Validation, Early Stopping, LR Scheduling

The lecture training loop trained on the full training set for a fixed number of epochs with no way to know whether the model was overfitting. In practice, we use a validation set which is not exposed to the model during the training phase (similar to the test dataset). After each epoch, we check how the performance of the model changed on this validation set and optionally take actions like reducing the learning rate or stopping training entirely if the model does not improve on the validation set anymore. 

Here you will implement the training loop with added validation logic it in three steps.

### 1a · Create a validation split

Use `random_split` (already imported) to split `full_train_dataset` into a training part (90 %) and a validation part (10 %). Then create a `DataLoader` for each.

In [ ]:
VAL_FRACTION = 0.1
n_val   = int(len(full_train_dataset) * VAL_FRACTION)
n_train = len(full_train_dataset) - n_val

train_dataset, val_dataset = TODO("split full_train_dataset into train_dataset and val_dataset using random_split")

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader   = TODO("create a DataLoader for val_dataset (no shuffling needed)")

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

### 1b · Add a validation loop

Below is the lecture training loop. Extend it by adding a **validation loop** after each epoch that computes the average loss on `val_dataloader`.

A few things to keep in mind:
- During validation the model should **not** update its weights.
- Dropout should be turned **off** during validation. Check the PyTorch docs for the right method to call on the model.
- Plot **both** `train_loss` and `val_loss` on the same axes so you can see whether the model is overfitting.

The `TODO()` calls below mark exactly where you need to add or change code.

In [ ]:
def evaluate_model(model, dataloader, loss_fn):
    TODO("put the model into eval mode to turn off dropout")
    
    total_loss = 0.0
    TODO("wrap the validation loop in torch.no_grad() to deactivate gradient calculations, which are only needed to update the weights of the model")
    for _, token_ids, label in dataloader:
        token_ids, label = token_ids.to(DEVICE), label.to(DEVICE)
        TODO("compute the validation loss for this batch and accumulate it in val_loss")
        
    TODO("Return average loss. Hint: You can use len(dataloader) to get the number of samples in the dataloader")

In [ ]:
model     = TextClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
optimiser = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()

EPOCHS = 50
train_losses = []
val_losses   = []

fig, ax = plt.subplots(figsize=(8, 4))

for epoch in range(EPOCHS):

    # --- Training ---
    model.train()
    epoch_loss = 0.0
    for _, token_ids, label in train_dataloader:
        token_ids, label = token_ids.to(DEVICE), label.to(DEVICE)
        loss = loss_fn(model(token_ids), label)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_dataloader))

    # --- Validation ---
    val_loss = evaluate_model(model, val_dataloader, loss_fn)
    val_losses.append(val_loss)

    # Live plot
    if (epoch + 1) % 5 == 0 or epoch == 0:
        ax.cla()
        ax.plot(train_losses, label="train")
        TODO("add a second line for val_losses to the plot")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.set_title(f"Epoch {epoch+1}/{EPOCHS}")
        display.clear_output(wait=True)
        display.display(fig)
        print(f"Epoch {epoch+1:>3}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")

plt.close()

### 1c · Early stopping and LR scheduling

Now add two more improvements to the loop above:

**Early stopping:** keep track of the best validation loss seen so far. If it does not improve for `PATIENCE` epochs in a row, stop training and print a message.

**LR scheduling:** use `torch.optim.lr_scheduler.ReduceLROnPlateau` to halve the learning rate whenever the validation loss has not improved for 5 epochs.

Both of these require the validation loss you computed in 1b — make sure that loop is working before you continue.

> **Hint:** `ReduceLROnPlateau` is initialised with `scheduler = ReduceLROnPlateau(optimiser, ...)` and updated with `scheduler.step(val_loss)` after the validation loop. Check the PyTorch docs for the right arguments.

In [ ]:
model     = TextClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)
optimiser = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()
scheduler = TODO("initialise ReduceLROnPlateau: halve lr when val loss doesn't improve for 5 epochs")

EPOCHS  = 100
PATIENCE = 10

train_losses = []
val_losses   = []
best_val_loss   = TODO("what value should best_val_loss start at?")
epochs_no_improve = 0

fig, ax = plt.subplots(figsize=(8, 4))

for epoch in range(EPOCHS):

    # --- Training (unchanged) ---
    model.train()
    epoch_loss = 0.0
    for _, token_ids, label in train_dataloader:
        token_ids, label = token_ids.to(DEVICE), label.to(DEVICE)
        loss = loss_fn(model(token_ids), label)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_dataloader))

    # --- Validation ---
    val_loss = evaluate_model(model, val_dataloader, loss_fn)
    val_losses.append(val_loss)

    # --- LR scheduler step ---
    TODO("call scheduler.step with the current validation loss")

    # --- Early stopping ---
    if TODO("check whether val_loss is better than best_val_loss. Hint: we typically want to minimize the loss!"):
        best_val_loss     = TODO("update best_val_loss")
        epochs_no_improve = TODO("reset the patience counter")
    else:
        epochs_no_improve = TODO("increment the patience counter")

    if TODO("check whether patience (defined in variable PATIENCE) is exhausted"):
        print(f"Early stopping at epoch {epoch+1}.")
        break

    # Live plot
    if (epoch + 1) % 5 == 0 or epoch == 0:
        ax.cla()
        ax.plot(train_losses, label="train")
        ax.plot(val_losses, label="val")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.set_title(f"Epoch {epoch+1}/{EPOCHS}")
        display.clear_output(wait=True)
        display.display(fig)
        print(f"Epoch {epoch+1:>3}  train={train_losses[-1]:.4f}  val={val_loss:.4f}  patience={epochs_no_improve}/{PATIENCE}")

plt.close()

#### Question: 

- How do you detect if the model is overfitting. Ask an AI of your choice, what are the three most common approaches in DL to avoid overfitting. Do we apply any of these techniques here?
- At what epoch did training stop? Did the train and val curves diverge before that point? What does this tell you about the model?

---

## Task 2 · Token-Level Predictions

So far the classifier produces **one** label for the **whole** sequence (many-to-one). Next week we will look at Named Entity Recognition, where the model must produce a label for **every token** (many-to-many).

The `RNNEncoder` already returns `output` — a tensor of shape `(batch, seq_len, hidden_dim)` containing the hidden state at every time step. All that needs to change is the classification head: instead of applying a linear layer to the final hidden state `h_T`, we apply it to **every** position in `output`.

Fill in the `forward` method of `TokenClassifier` below.

In [ ]:
class TokenClassifier(nn.Module):
    """
    Applies the classification head at every time step instead of only
    at the final hidden state — turning the many-to-one model into a
    many-to-many model.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.rnn        = RNNEncoder(vocab_size, embed_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, token_ids):
        """
        Parameters
        ----------
        token_ids : LongTensor (batch, seq_len)

        Returns
        -------
        logits : FloatTensor (batch, seq_len, num_classes)
        """
        output, h_T = self.rnn(token_ids)   # output: (batch, seq_len, hidden_dim)
        # HINT: PyTorch's nn.Linear natively works with a sequence dimension and 
        # automatically applies the transformation to all elements in the batch 
        # and seq dimension independently.
        return TODO("apply self.classifier to every position in output")

In [ ]:
# Verify the output shape
token_clf = TokenClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(DEVICE)

_, sample_ids, _ = full_train_dataset[0]
sample_ids = sample_ids.unsqueeze(0).to(DEVICE)  # add batch dimension

with torch.no_grad():
    logits = token_clf(sample_ids)

print("Output shape:", logits.shape)
assert logits.shape == (1, MAX_SEQ_LEN, NUM_CLASSES), \
    f"Expected (1, {MAX_SEQ_LEN}, {NUM_CLASSES}), got {logits.shape}"
print("Shape check passed ✓")

#### Question: 

The model outputs a prediction at every token position, including padding positions. Why is that a problem, and how might you deal with it during training? **Hint:** Check the documentation of nn.CrossEntropyLoss() and especially the ignore_index parameter

---

## Task 3 · Stacked RNN

In the lecture, we built a single-layer RNN. In modern natural language processing and Information Extraction architectures, models often stack multiple RNN layers on top of each other. The output hidden state of the first layer at time step $t$ becomes the input to the second layer at time step $t$, and so on.

To build custom deep RNN architectures, PyTorch provides `nn.ModuleList` and `nn.RNNCell`. 
- `nn.RNNCell(input_size, hidden_size)` handles exactly **one time step** for **one layer**.
- `nn.ModuleList` is a special PyTorch container. If you store layers in a regular Python list (e.g., `[layer1, layer2]`), PyTorch will not know they exist, meaning `.to(device)` won't work on them and their parameters won't update during training! Wrapping them in a `nn.ModuleList` registers them correctly.

Look at the structural diagram below to see how hidden states flow horizontally across time steps and vertically across layers:



**Your Task:**
Complete the implementation of the `MultiLayerRNNEncoder` below. You need to:
1. Initialize the hidden states for **all layers** as a list of zero-tensors.
2. Complete the vertical layer loop inside the time loop to compute the hidden state updates.
3. Make sure the padding mask is applied across all layers so that padding tokens don't update any hidden states.

Below is a visualization of a deep RNN with many layers. The general idea: The **output** of the $i$-th RNN Cell of the $j$-th layer is the **input** of the $i$-th RNN Cell of the $j+1$-th layer

In [ ]:
from IPython.display import Image, display

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/f/f9/Deep_RNN_architecture.svg/330px-Deep_RNN_architecture.svg.png"
display(Image(url=url))

In [ ]:
class MultiLayerRNNEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.1):
        super(MultiLayerRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # We use nn.ModuleList so PyTorch tracks the parameters of every single layer
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            # Layer 0 takes the text embedding size as input.
            # Layer 1 and beyond take the hidden state size from the layer below.
            input_size = embed_dim if i == 0 else hidden_dim
            self.layers.append(nn.RNNCell(input_size, hidden_dim, nonlinearity='tanh'))
            
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len = x.shape
        embedded = self.dropout(self.embedding(x)) 
        
        # Hint: You need a list containing 'num_layers' tensors, each of shape (batch_size, hidden_dim)
        h_ts = [TODO("Initialize the hidden state h_0 for every RNN layer in this line") for _ in range(self.num_layers)] 
        
        all_hidden_states = []
        
        # Horizontally iterate over every time step (sequence length)
        for t in range(seq_len):
            mask = (x[:, t] != 0).float().unsqueeze(1)
            
            # The initial input to the stacked network at time t is the token's embedding vector
            current_input = TODO("set the embedding of the current token as input to the first RNN layer")
            
            # Vertically iterate up through the stacked layers
            for l in range(self.num_layers):
                # Compute the proposed next hidden state for layer l
                current_layer = self.layers[l]
                # HINT: the_new_hidden_state = self.layers[layer_index](current_layer_input, previous_hidden_state_of_this_layer)
                h_t_next = current_layer(TODO("Pass the correct input to the current RNN layer"))
                
                # Conditionally update: if mask is 0, freeze and retain old h_ts[l]
                h_ts[l] = mask * h_t_next + (1.0 - mask) * h_ts[l]
                
                # The output of this layer becomes the input to the next layer up
                current_input = h_ts[l]
                
            # We collect the hidden state of the TOPMOST layer to build our sequence representations
            all_hidden_states.append(h_ts[-1].unsqueeze(1))
        
        output = torch.cat(all_hidden_states, dim=1)
        return output, h_ts[-1]

In [ ]:
# Verify the stacked model
stacked_model = MultiLayerRNNEncoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, num_layers=2).to(DEVICE)

with torch.no_grad():
    output, h_T = stacked_model(sample_ids)

print("Output sequence shape:", output.shape)
print("Final hidden state shape:", h_T.shape)

assert output.shape == (1, MAX_SEQ_LEN, HIDDEN_DIM), f"Expected (1, {MAX_SEQ_LEN}, {HIDDEN_DIM}), got {output.shape}"
assert h_T.shape == (1, HIDDEN_DIM), f"Expected (1, {HIDDEN_DIM}), got {h_T.shape}"
print("Shape check passed ✓")